# 00 - Validation Map

This notebook is the entry point for the `capital_run_v1` model-validation notebook pack. It maps each notebook to the committed synthetic fixture artifacts, golden reconciliation checks, and prototype NPR 2.0 proposed-rule assumptions.

Prototype caution: the fixture is deterministic synthetic development data. The notebooks are model-validation evidence for this prototype only and do not present final regulatory capital.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from IPython.display import Markdown, display

from examples.capital_run_fixture import load_capital_run_v1_fixture, policy_from_fixture

fixture = load_capital_run_v1_fixture()
policy = policy_from_fixture(fixture)
manifest = fixture.manifest
expected = fixture.expected_outputs
params = fixture.params


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


def scalar_value(name: str) -> str:
    return f"{float(expected['scalars'][name]['value']):,.6f}"


display(
    Markdown(
        f"Loaded fixture `{params['run_id']}` for desk `{params['desk_id']}` "
        f"under `{params['regime']}`. Schema `{params['schema_version']}`, "
        f"generator version `{params['generator_version']}`, seed `{params['seed']}`."
    )
)

## Notebook Map

The pack is ordered as a validation narrative: evidence and calibration inputs first, capital assembly last. Notebook `00` is a guide; notebooks `01` through `06` contain the executable validation views.

In [ ]:
notebook_rows = [
    [
        "00_validation_map.ipynb",
        "Pack index and fixture lineage",
        "manifest.json, params.json, expected_outputs.json",
        "not applicable",
    ],
    [
        "01_rfet_evidence_classification.ipynb",
        "RFET evidence and classification",
        "risk_factors.csv, rfet_observations.csv, expected_outputs.json",
        "classifications",
    ],
    [
        "02_stress_period_selection.ipynb",
        "Stress-period selection",
        "stress_history_metadata.csv, stress_histories.npz, expected_outputs.json",
        "selected_stress_periods",
    ],
    [
        "03_lha_es_imcc.ipynb",
        "Nested liquidity-horizon ES and IMCC",
        "scenario_cube.npz, scenario_metadata.csv, risk_factors.csv, expected_outputs.json",
        "unconstrained_lha_es, constrained_lha_es, imcc",
    ],
    [
        "04_nmrf_chain.ipynb",
        "NMRF method selection to SES",
        "nmrf_evidence.json, nmrf_artifacts.npz, stress histories, expected_outputs.json",
        "nmrf_methods, reconciliation, total_ses",
    ],
    [
        "05_pla_backtesting.ipynb",
        "PLA and backtesting",
        "pla_bt_vectors.npz, expected_outputs.json",
        "pla, backtesting, pla_ks_statistic, supervisory_multiplier",
    ],
    [
        "06_capital_assembly.ipynb",
        "Models-based capital assembly",
        "scenario_cube.npz, expected_outputs.json",
        "models_based_capital, capital.binding_term",
    ],
]
display(markdown_table(["Notebook", "Validation purpose", "Fixture inputs", "Golden checks"], notebook_rows))

## Proposed-Rule Parameter Map

These assumptions are deliberately surfaced in the pack because the repository is a transparent prototype, not an implementation of final U.S. regulatory capital requirements.

In [ ]:
assumption_rows = [
    ["Regime", str(params["regime"]), "Primary prototype profile; not final regulatory capital"],
    ["Expected shortfall", f"{policy.es_confidence_level:.1%}", "Applied to supplied scenario or stress loss vectors"],
    ["Liquidity horizon adjustment", "Nested LH vectors", "Risk-factor subsets, not final-scalar scaling"],
    ["RFET short-LH threshold", str(policy.rfet_short_lh_threshold), "For LH <= 20 days"],
    ["RFET long-LH threshold", str(policy.rfet_long_lh_threshold), "For LH > 20 days"],
    ["NMRF taxonomy", "MODELLABLE / TYPE_A_NMRF / TYPE_B_NMRF", "Fed NPR 2.0 proposed rule (see regulatory anchors)"],
    ["Type A NMRFs", "IMCC and SES", "Routed to both populations"],
    ["Type B NMRFs", "SES only", f"Type B rho = {policy.type_b_ses_rho:.2f}"],
    ["PLA statistic", "Kolmogorov-Smirnov", "HPL versus RTPL prototype gate"],
    ["Backtesting", "APL and HPL exceptions", "250-business-day fixture window at 97.5% and 99.0% VaR"],
]
display(markdown_table(["Area", "Fixture setting", "Validation note"], assumption_rows))

## Golden Output Reconciliation Inventory

The table below is a compact index of outputs reconciled across the executable notebooks. Golden values are deterministic fixture drift gates, not independent regulatory benchmarks.

In [ ]:
classification_counts = Counter(expected["classifications"].values())
selected_periods = expected["selected_stress_periods"]
backtesting = expected["backtesting"]

summary_rows = [
    ["RFET classifications", ", ".join(f"{key}: {classification_counts[key]}" for key in sorted(classification_counts)), "01"],
    ["Selected stress periods", f"{len(selected_periods)} risk classes", "02"],
    ["Unconstrained LHA ES", scalar_value("unconstrained_lha_es"), "03"],
    ["Constrained LHA ES", scalar_value("constrained_lha_es"), "03"],
    ["IMCC", scalar_value("imcc"), "03, 06"],
    ["NMRF methods", ", ".join(f"{name}: {method}" for name, method in sorted(expected["nmrf_methods"].items())), "04"],
    ["NMRF reconciliation", f"passed={expected['reconciliation']['passed']}; artifacts={expected['reconciliation']['artifact_count']}", "04"],
    ["Total SES", scalar_value("total_ses"), "04, 06"],
    ["PLA", f"zone={expected['pla']['zone']}; KS={scalar_value('pla_ks_statistic')}", "05"],
    ["Backtesting", f"eligible={backtesting['model_eligible']}; window={backtesting['window_size']}", "05"],
    ["Supervisory multiplier", scalar_value("supervisory_multiplier"), "05, 06"],
    ["Models-based capital", f"{scalar_value('models_based_capital')}; binding={expected['capital']['binding_term']}", "06"],
]
display(markdown_table(["Output", "Golden fixture value", "Notebook"], summary_rows))

## Fixture Lineage and Sign Conventions

The canonical fixture is committed under `tests/fixtures/capital_run_v1`. The loader verifies SHA-256 checksums from `manifest.json` before notebooks consume the data.

In [ ]:
file_rows = []
for filename, metadata in sorted(manifest["files"].items()):
    file_rows.append([filename, str(metadata["sha256"])[:12] + "..."])
display(markdown_table(["Fixture file", "SHA-256 prefix"], file_rows))

sign_rows = []
for filename, conventions in sorted(manifest["sign_conventions"].items()):
    for field, convention in sorted(conventions.items()):
        sign_rows.append([filename, field, convention])
display(markdown_table(["Array file", "Field", "Sign convention"], sign_rows))

## Open Prototype Gaps

The validation pack intentionally stops at the ex-post IMA-style capital assembly demonstrated by the fixture. Known gaps remain out of scope for this notebook suite:

- SBM/DRC/RRAO fallback stack, CVA, and firm/legal-entity consolidation.
- Trading-desk approval lifecycle and supervisory workflow evidence.
- Real market-data sourcing, vendor adapters, and proprietary instrument classification.
- Production direct, stepwise, and full-revaluation NMRF pricing engines.
- Full business-calendar governance beyond fixture dates and optional holiday masks.
- Regulatory disclosure templates and large-run storage or telemetry integrations.